# Slice-Aware Scheduling

This notebook demonstrates slice-aware scheduling in Sionna's system-level simulator.
UEs are partitioned into network slices (eMBB and URLLC) using a QFI (QoS Flow Indicator),
and each slice receives a dedicated PRB quota with a per-slice scheduling policy.

We compare three scenarios:
1. **Baseline** – no slicing, standard `PFSchedulerSUMIMO`
2. **Equal split** – 50 / 50 PRB split between eMBB (PF) and URLLC (priority)
3. **URLLC-favoured** – 80 % of PRBs reserved for URLLC

For each scenario we measure per-slice E2E round-trip delay, throughput, and BLER.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from sionna.sys import (
    SliceAwarePFScheduler,
    validate_slice_config,
    get_slice_prb_ranges,
    build_slice_prb_mask,
)

# Use CPU for portability; switch to 'cuda' if a GPU is available
device = 'cpu'

## 1  Simulation helpers

The cell below provides lightweight delay-metric helpers that work directly
on the `hist` dictionary returned by `SystemLevelSimulator`.

In [ ]:
def compute_scheduling_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Mean inter-scheduling gap per UE (ms)."""
    is_scheduled = ~np.isnan(hist['harq'])  # [num_slots, num_bs, num_ut]
    delays = np.full((num_bs, num_ut_per_sector), np.nan)
    for bs in range(num_bs):
        for ue in range(num_ut_per_sector):
            sched_slots = np.where(is_scheduled[:, bs, ue])[0]
            if len(sched_slots) > 1:
                delays[bs, ue] = np.mean(np.diff(sched_slots)) * slot_duration_ms
    return delays


def compute_transmission_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Mean transmission delay per UE accounting for HARQ retransmissions (ms)."""
    harq = hist['harq']  # [num_slots, num_bs, num_ut]
    delays = np.full((num_bs, num_ut_per_sector), np.nan)
    for bs in range(num_bs):
        for ue in range(num_ut_per_sector):
            harq_seq = harq[:, bs, ue]
            sched_slots = np.where(~np.isnan(harq_seq))[0]
            if len(sched_slots) == 0:
                continue
            tb_delays, nack_count = [], 0
            for s in sched_slots:
                if harq_seq[s] == 0:
                    nack_count += 1
                elif harq_seq[s] == 1:
                    tb_delays.append((1 + nack_count) * slot_duration_ms)
                    nack_count = 0
            if tb_delays:
                delays[bs, ue] = np.mean(tb_delays)
    return delays


def compute_e2e_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Estimated E2E round-trip delay per UE (ms, symmetric UL/DL assumption)."""
    sched = compute_scheduling_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms)
    tx    = compute_transmission_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms)
    one_way = np.nansum([sched, tx], axis=0)
    return 2.0 * one_way, sched, tx

## 2  QFI assignment and slice configurations

We create a simple two-slice scenario:
- **QFI 1** → eMBB slice (proportional-fair scheduling)
- **QFI 9** → URLLC slice (priority / highest-rate-first scheduling)

In [ ]:
num_ut_per_sector = 10
num_freq_res      = 52   # PRBs (matches a standard 10 MHz NR carrier)
num_ofdm_sym      = 14

# Assign first 5 UEs to eMBB (QFI=1), remaining 5 to URLLC (QFI=9)
qfi_per_ut = torch.cat([
    torch.full((5,), 1, dtype=torch.int32),
    torch.full((5,), 9, dtype=torch.int32),
])
print("QFI assignment:", qfi_per_ut.tolist())

# --- Slice configurations to sweep ---
sweep_configs = {
    "equal_split": {
        1: {"num_prbs": 26, "policy": "proportional_fair"},
        9: {"num_prbs": 26, "policy": "priority"},
    },
    "favor_urllc": {
        1: {"num_prbs": 10, "policy": "proportional_fair"},
        9: {"num_prbs": 42, "policy": "priority"},
    },
}

# Validate all configs
for name, cfg in sweep_configs.items():
    validate_slice_config(qfi_per_ut, cfg, num_freq_res)
    print(f"Config '{name}' validated OK")

## 3  PRB mask inspection

Visualise which PRBs belong to which slice for the equal-split config.

In [ ]:
cfg = sweep_configs["equal_split"]
slice_ranges = get_slice_prb_ranges({q: c["num_prbs"] for q, c in cfg.items()})
print("PRB ranges:", slice_ranges)

mask = build_slice_prb_mask(num_freq_res, qfi_per_ut, slice_ranges)
print(f"PRB mask shape: {mask.shape}  (num_freq_res × num_ut)")

plt.figure(figsize=(10, 3))
plt.imshow(mask.numpy().T, aspect="auto", origin="lower", cmap="Blues")
plt.xlabel("PRB index")
plt.ylabel("UE index")
plt.title("PRB allocation mask (equal split)")
plt.colorbar(label="allowed")
plt.tight_layout()
plt.show()

## 4  Standalone scheduler smoke-test

We run a quick sanity check on `SliceAwarePFScheduler` without the full
`SystemLevelSimulator` to confirm that slice isolation is enforced.

In [ ]:
slice_cfg = sweep_configs["equal_split"]

scheduler = SliceAwarePFScheduler(
    qfi_per_ut=qfi_per_ut,
    slice_prb_config=slice_cfg,
    num_ut=num_ut_per_sector,
    num_freq_res=num_freq_res,
    num_ofdm_sym=num_ofdm_sym,
    device=device,
)

rate_last   = torch.rand(num_ut_per_sector) * 100
rate_ach    = torch.rand(num_ofdm_sym, num_freq_res, num_ut_per_sector) * 100
is_sched    = scheduler(rate_last, rate_ach)   # [sym, prb, ut, streams]

print(f"is_scheduled shape: {is_sched.shape}")

# --- Verify slice isolation ---
is_sched_np = is_sched.squeeze(-1).numpy()   # [sym, prb, ut]
slice_ranges = scheduler.slice_ranges

for qfi_val, (start, end) in slice_ranges.items():
    ut_mask_np = (qfi_per_ut == qfi_val).numpy()
    # UEs NOT in this slice must NOT be scheduled on its PRBs
    out_of_slice = is_sched_np[:, start:end, :][:, :, ~ut_mask_np]
    assert not out_of_slice.any(), f"Slice isolation violated for QFI {qfi_val}!"

print("Slice isolation check: PASSED")

# --- Verify exactly one UE per PRB ---
per_prb_sum = is_sched.sum(dim=-2).squeeze(-1)   # [sym, prb]
assert (per_prb_sum == 1).all(), "Not exactly one UE per PRB!"
print("Single-UE-per-PRB check: PASSED")

## 5  Integration with `SystemLevelSimulator`

When running the full simulator, pass `qfi_per_ut` and `slice_prb_config`
to the constructor.  The simulator instantiates `SliceAwarePFScheduler`
in place of the default `PFSchedulerSUMIMO`.

```python
from sionna.sys import SliceAwarePFScheduler, validate_slice_config

# --- (re-)build the simulator with slice-aware scheduling ---
if qfi_per_ut is not None and slice_prb_config is not None:
    validate_slice_config(qfi_per_ut, slice_prb_config, resource_grid.fft_size)
    sls.scheduler = SliceAwarePFScheduler(
        qfi_per_ut=qfi_per_ut,
        slice_prb_config=slice_prb_config,
        num_ut=sls.num_ut_per_sector,
        num_freq_res=resource_grid.fft_size,
        num_ofdm_sym=resource_grid.num_ofdm_symbols,
        batch_size=[sls.batch_size, sls.num_bs],
        num_streams_per_ut=sls.num_streams_per_ut,
        device=sls.device,
    )

# Run simulation
hist = sls(num_slots, alpha_ul, p0_dbm_ul, bler_target, olla_delta_up)
```

After the simulation, compute per-slice KPIs as shown in Section 6.

## 6  KPI computation and comparison

Once you have `hist` from a simulator run, use the helpers defined in
Section 2 to compute E2E round-trip delay per UE, then group by QFI.

In [ ]:
# Example: compare per-slice KPIs from two hist dicts obtained via the
# SystemLevelSimulator (replace hist_baseline / hist_sliced with real data)

# ---- synthetic placeholder data for illustration ----
num_slots  = 200
num_bs     = 1
slot_ms    = 1.0   # 1 ms slot (15 kHz SCS)

rng = np.random.default_rng(0)

def _make_hist(harq_prob_per_ut):
    """Generate synthetic hist dict for demonstration."""
    # Schedule each UE with probability proportional to its index
    harq = np.full((num_slots, num_bs, num_ut_per_sector), np.nan)
    for s in range(num_slots):
        for u, p in enumerate(harq_prob_per_ut):
            if rng.random() < p:
                harq[s, 0, u] = float(rng.random() > 0.1)  # 90 % ACK rate
    return {"harq": harq}

# Baseline: uniform scheduling probability
hist_baseline = _make_hist([0.3] * num_ut_per_sector)

# Slice-aware: URLLC UEs (5-9) scheduled more frequently
hist_sliced = _make_hist([0.2, 0.2, 0.2, 0.2, 0.2,   # eMBB
                          0.5, 0.5, 0.5, 0.5, 0.5])   # URLLC

# ---- compute delays ----
e2e_base, sched_base, tx_base = compute_e2e_delay(
    hist_baseline, num_slots, num_bs, num_ut_per_sector, slot_ms)
e2e_sliced, sched_sliced, tx_sliced = compute_e2e_delay(
    hist_sliced, num_slots, num_bs, num_ut_per_sector, slot_ms)

qfi_np = qfi_per_ut.numpy()
slice_names = {1: "eMBB (QFI=1)", 9: "URLLC (QFI=9)"}

print(f"{'Slice':<20} {'Baseline E2E RTT':>18} {'Slice-aware E2E RTT':>22}")
print("-" * 62)
for qfi_val in sorted(np.unique(qfi_np)):
    ue_mask = qfi_np == qfi_val
    b_mean  = np.nanmean(e2e_base[:, ue_mask])
    s_mean  = np.nanmean(e2e_sliced[:, ue_mask])
    name    = slice_names.get(qfi_val, f"QFI={qfi_val}")
    print(f"{name:<20} {b_mean:>16.2f} ms {s_mean:>20.2f} ms")

In [ ]:
# ---- Plot: E2E RTT CDF per slice, baseline vs. slice-aware ----

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
slice_colors = {1: "steelblue", 9: "tomato"}

for ax, (scenario_name, e2e) in zip(
        axes, [("Baseline", e2e_base), ("Slice-aware", e2e_sliced)]):
    for qfi_val in sorted(np.unique(qfi_np)):
        ue_mask = qfi_np == qfi_val
        vals = e2e[:, ue_mask].ravel()
        vals = vals[~np.isnan(vals)]
        sorted_vals = np.sort(vals)
        cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
        ax.plot(sorted_vals, cdf,
                color=slice_colors[qfi_val],
                label=slice_names.get(qfi_val, f"QFI={qfi_val}"))
    ax.set_title(scenario_name)
    ax.set_xlabel("E2E round-trip delay (ms)")
    ax.set_ylabel("CDF")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("E2E RTT CDF per slice")
plt.tight_layout()
plt.show()

## 7  PRB sweep across configurations

To sweep over multiple PRB splits, instantiate the scheduler once per
configuration and collect `hist` for each run.

In [ ]:
urllc_prb_fractions = [0.2, 0.4, 0.5, 0.6, 0.8]
urllc_mean_e2e = []
embb_mean_e2e  = []

for frac in urllc_prb_fractions:
    n_urllc = max(1, round(num_freq_res * frac))
    n_embb  = num_freq_res - n_urllc
    if n_embb < 1:
        continue

    cfg = {
        1: {"num_prbs": n_embb,  "policy": "proportional_fair"},
        9: {"num_prbs": n_urllc, "policy": "priority"},
    }

    sched = SliceAwarePFScheduler(
        qfi_per_ut=qfi_per_ut,
        slice_prb_config=cfg,
        num_ut=num_ut_per_sector,
        num_freq_res=num_freq_res,
        num_ofdm_sym=num_ofdm_sym,
        device=device,
    )

    # In a real run, replace synthetic hist with output from SystemLevelSimulator
    p_urllc = 0.1 + 0.8 * frac     # simplified model: more PRBs → higher sched prob
    p_embb  = 0.1 + 0.8 * (1 - frac)
    hist_sweep = _make_hist([p_embb] * 5 + [p_urllc] * 5)

    e2e, _, _ = compute_e2e_delay(
        hist_sweep, num_slots, num_bs, num_ut_per_sector, slot_ms)

    urllc_mean_e2e.append(np.nanmean(e2e[:, qfi_np == 9]))
    embb_mean_e2e.append(np.nanmean(e2e[:, qfi_np == 1]))

plt.figure(figsize=(7, 4))
plt.plot(urllc_prb_fractions, urllc_mean_e2e, 'o-', color="tomato",     label="URLLC (QFI=9)")
plt.plot(urllc_prb_fractions, embb_mean_e2e,  's--', color="steelblue", label="eMBB  (QFI=1)")
plt.xlabel("URLLC PRB fraction")
plt.ylabel("Mean E2E RTT (ms)")
plt.title("Per-slice E2E delay vs. PRB allocation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Component | Location |
|-----------|----------|
| `SliceAwarePFScheduler` | `sionna.sys` |
| `build_slice_prb_mask` | `sionna.sys` |
| `validate_slice_config` | `sionna.sys` |
| `get_slice_prb_ranges` | `sionna.sys` |

To use slice-aware scheduling in a full simulation:
1. Construct `qfi_per_ut` (one QFI integer per UE)
2. Build `slice_prb_config` (QFI → `{num_prbs, policy}`)
3. Call `validate_slice_config` to catch configuration errors early
4. Instantiate `SliceAwarePFScheduler` and assign it to `sls.scheduler`
5. Run the simulator and use `compute_e2e_delay` to compare per-slice KPIs